# Fine-tuning DistilBERT for Sarcasm Detection

Trains the model served by `app.py` in the Sarcasm-Detection repo.

**Set the runtime to a GPU before running:** `Runtime -> Change runtime type -> T4 GPU`.
The whole notebook takes roughly 5 minutes on a T4.

### What this replaces
The 2020 original used a Bidirectional LSTM with Attention over ELMo vectors. ELMo's TF1
`hub.Module` API no longer exists, and the published weights were trained on a Twitter
corpus that isn't in the repo, so the model had to be rebuilt regardless.

### Read this before trusting the accuracy number
In this dataset **every sarcastic headline is from The Onion and every non-sarcastic one is
from HuffPost**. Some of what the model learns is house style, not sarcasm. The
`article_link` column is dropped for exactly this reason (the domain *is* the label), but
the writing style itself still correlates. Treat the result as *accuracy on this dataset*,
not as general-purpose sarcasm detection.

## 1. Install pinned dependencies

Pinned to the same versions as `requirements.txt`, so the artefacts this notebook produces
load cleanly in the app.

In [ ]:
!pip install -q 'transformers==5.14.1' 'tokenizers==0.22.2' 'safetensors==0.8.0' \
                'huggingface_hub==1.24.0' 'datasets>=3.0' 'accelerate>=1.0' \
                'scikit-learn==1.9.0' 'pandas==2.2.2'

import torch
print('torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - switch the runtime to T4 or training will take hours')

## 2. Configuration

Set `REPO_URL` to your GitHub fork so the notebook can pull both the raw data and
`cleaning.py`. Using the repo's own cleaning function is what guarantees training and
serving preprocess text identically.

Set `HF_REPO_ID` to where you want the trained weights published, e.g.
`your-username/distilbert-sarcasm-headlines`.

In [ ]:
REPO_URL    = ''   # e.g. 'https://github.com/your-username/Sarcasm-Detection.git'
HF_REPO_ID  = ''   # e.g. 'your-username/distilbert-sarcasm-headlines'

BASE_MODEL    = 'distilbert-base-uncased'
MAX_LENGTH    = 48     # must match MAX_LENGTH in Code/pkg/model_training/transformer.py
BATCH_SIZE    = 64
LEARNING_RATE = 2e-5
EPOCHS        = 3
SEED          = 42

## 3. Get the repo (data + the shared cleaning function)

If you leave `REPO_URL` blank you'll be prompted to upload the two
`Sarcasm_Headlines_Dataset*.json` files by hand, and the notebook falls back to an inlined
copy of the cleaning preset.

In [ ]:
import os, sys, glob

REPO_DIR = None
if REPO_URL:
    if not os.path.isdir('repo'):
        !git clone --depth 1 $REPO_URL repo
    REPO_DIR = 'repo'
    sys.path.insert(0, REPO_DIR)
    from Code.pkg.data_processing.cleaning import clean_for_model
    RAW_DIR = os.path.join(REPO_DIR, 'Code/pkg/datasets/news_headlines/raw_data')
    print('Using cleaning.py from the repo (single source of truth).')
else:
    from google.colab import files
    print('Upload Sarcasm_Headlines_Dataset.json and Sarcasm_Headlines_Dataset_v2.json')
    files.upload()
    RAW_DIR = '.'
    # Fallback copy of Code/pkg/data_processing/cleaning.py::clean_for_model.
    # Keep in sync if you change the original.
    import re
    def clean_for_model(x: str) -> str:
        x = re.sub(r'http\S+', '', x) + ' '
        x = x.lower()
        x = re.sub(r'[^\x00-\x7F]+', ' ', x)
        x = re.sub(r'([@][\w_-]+)', '<user>', x)
        x = x.replace('#sarcasm', ' ').replace('#not', 'not')
        x = re.sub(r'([#][\w_-]+)', ' ', x)
        return ' '.join(x.split()).strip()

print(sorted(glob.glob(os.path.join(RAW_DIR, '*.json'))))

## 4. Load, merge and de-duplicate

v1 and v2 overlap heavily, so they're merged and de-duplicated on the headline text.
**De-duplication happens before the split** - otherwise the same headline lands in both
train and test and inflates the score.

In [ ]:
import pandas as pd

frames = []
for name in ['Sarcasm_Headlines_Dataset.json', 'Sarcasm_Headlines_Dataset_v2.json']:
    path = os.path.join(RAW_DIR, name)
    if os.path.isfile(path):
        frame = pd.read_json(path, lines=True)[['is_sarcastic', 'headline']]
        print(name, len(frame), 'rows')
        frames.append(frame)

# Columns are selected by NAME: v1 and v2 store the same keys in a different order, so
# positional renaming silently mislabels v1.
data = pd.concat(frames, ignore_index=True).rename(
    columns={'is_sarcastic': 'label', 'headline': 'text'})

# article_link is never loaded - the domain in the URL is the label.
data['text'] = data['text'].astype(str).map(clean_for_model)
data = data[data['text'].str.len() > 0]
data = data.drop_duplicates(subset='text').reset_index(drop=True)

print()
print(len(data), 'unique headlines')
print(data['label'].value_counts().rename({0: 'non-sarcastic', 1: 'sarcastic'}))
data.head()

## 5. Stratified 80/10/10 split

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    data, test_size=0.2, stratify=data['label'], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

for name, frame in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print('%-6s %6d   sarcastic=%.3f' % (name, len(frame), frame['label'].mean()))

## 6. Baseline: TF-IDF + Logistic Regression

Gives the transformer's score something to be measured against.

Hyperparameters are taken from the repo's own `Code/pkg/model_training/MLmodels.py`
(`log_reg`: `C=10, max_iter=300`) and `create_vectors.py` (`max_df=0.1, max_features=5000`).
They're restated rather than imported because importing `MLmodels` pulls in
`helper.py -> create_vectors.py -> tensorflow_hub`, the dead TF1 chain.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

tfidf = TfidfVectorizer(max_df=0.1, max_features=5000)
X_train = tfidf.fit_transform(train_df['text'])
X_test = tfidf.transform(test_df['text'])

baseline = LogisticRegression(C=10, max_iter=300).fit(X_train, train_df['label'])
baseline_pred = baseline.predict(X_test)

BASELINE = {'accuracy': accuracy_score(test_df['label'], baseline_pred),
            'f1': f1_score(test_df['label'], baseline_pred)}
print('TF-IDF + LogReg baseline:', {k: round(v, 4) for k, v in BASELINE.items()})

## 7. Tokenise

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenise(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

def to_dataset(frame):
    dataset = Dataset.from_pandas(frame[['text', 'label']], preserve_index=False)
    return dataset.map(tokenise, batched=True, remove_columns=['text'])

train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)
print(train_ds)

## 8. Train

The API here is transformers **v5**: `eval_strategy` (not `evaluation_strategy`) and
`processing_class` (not `tokenizer`).

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, EarlyStoppingCallback)
from sklearn.metrics import (precision_recall_fscore_support, matthews_corrcoef,
                             confusion_matrix)

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0: 'not_sarcastic', 1: 'sarcastic'},
    label2id={'not_sarcastic': 0, 'sarcastic': 1})

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary')
    # The 2020 evaluate_model() reported everything except accuracy; we add it so the
    # headline claim is actually backed by a number.
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1,
            'precision': precision, 'recall': recall,
            'mcc': matthews_corrcoef(labels, preds)}

args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to='none',
    seed=SEED)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

trainer.train()

## 9. Evaluate on the held-out test set

If accuracy comes out materially above ~0.94, suspect a leak and re-check that
de-duplication ran before the split.

In [ ]:
import json

test_metrics = trainer.evaluate(test_ds, metric_key_prefix='test')
preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
cm = confusion_matrix(test_df['label'], preds)

print()
print('--- Test set ---')
for key in ['test_accuracy', 'test_f1', 'test_precision', 'test_recall', 'test_mcc']:
    print('%-16s %.4f' % (key, test_metrics[key]))
print()
print('Confusion matrix [[TN FP] [FN TP]]:')
print(cm)
print()
print('Baseline accuracy %.4f  ->  DistilBERT %.4f'
      % (BASELINE['accuracy'], test_metrics['test_accuracy']))

metrics = {
    'base_model': BASE_MODEL,
    'dataset': 'news_headlines (Misra) v1+v2 merged, de-duplicated',
    'n_total': int(len(data)), 'n_train': int(len(train_df)),
    'n_val': int(len(val_df)), 'n_test': int(len(test_df)),
    'distilbert': {key.replace('test_', ''): round(float(test_metrics[key]), 4)
                   for key in ['test_accuracy', 'test_f1', 'test_precision',
                               'test_recall', 'test_mcc']},
    'baseline_tfidf_logreg': {k: round(float(v), 4) for k, v in BASELINE.items()},
    'confusion_matrix': cm.tolist(),
    'hyperparameters': {'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
                        'learning_rate': LEARNING_RATE, 'epochs': EPOCHS, 'seed': SEED},
    'caveat': ('Sarcastic headlines are all from The Onion and non-sarcastic ones all '
               'from HuffPost, so part of this score reflects publication style rather '
               'than sarcasm itself.')}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print()
print('Wrote metrics.json')

## 10. Save locally and download

Unzip into `Code/pkg/trained_models/distilbert-sarcasm/` and the app picks it up with no
configuration at all. Also commit `metrics.json` to the repo root - the README results
table and the app both read it.

In [ ]:
SAVE_DIR = 'distilbert-sarcasm'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -qr distilbert-sarcasm.zip $SAVE_DIR
!du -sh distilbert-sarcasm.zip

from google.colab import files
files.download('distilbert-sarcasm.zip')
files.download('metrics.json')

## 11. Push to the HuggingFace Hub

This is what lets Streamlit Community Cloud serve the model without the weights ever
entering your git repo. You'll need a **write** token from
[hf.co/settings/tokens](https://huggingface.co/settings/tokens).

Afterwards set `SARCASM_MODEL_ID` to your repo id in the Streamlit app's secrets.

In [ ]:
if HF_REPO_ID:
    from huggingface_hub import notebook_login
    notebook_login()

    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print('Pushed. Set this in the Streamlit app secrets:')
    print('SARCASM_MODEL_ID = "' + HF_REPO_ID + '"')
else:
    print('HF_REPO_ID is blank - skipping. Set it in the config cell to publish.')